In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

In [2]:
df = pd.read_csv('dirty_cafe_sales.csv')
print(f"Размер: {df.shape}")
df.head()

Размер: (10000, 8)
<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9667 non-null   str  
 2   Quantity          9862 non-null   str  
 3   Price Per Unit    9821 non-null   str  
 4   Total Spent       9827 non-null   str  
 5   Payment Method    7421 non-null   str  
 6   Location          6735 non-null   str  
 7   Transaction Date  9841 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [5]:
print("Пропуски:")
missing = df.isnull().sum()
print(missing[missing > 0])
print(f"\nВсего пропусков: {df.isnull().sum().sum()}")

Пропуски:
Item                 333
Quantity             138
Price Per Unit       179
Total Spent          173
Payment Method      2579
Location            3265
Transaction Date     159
dtype: int64

Всего пропусков: 6826


In [6]:
df = df.replace(['ERROR', 'UNKNOWN', 'error', 'unknown'], np.nan)

In [8]:
print("Пропуски после замены:")
missing_after = df.isnull().sum()
print(missing_after[missing_after > 0])
print(f"\nВсего пропусков после замены: {df.isnull().sum().sum()}")

Пропуски после замены:
Item                 969
Quantity             479
Price Per Unit       533
Total Spent          502
Payment Method      3178
Location            3961
Transaction Date     460
dtype: int64

Всего пропусков после замены: 10082


In [9]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 8 columns):
 #   Column            Non-Null Count  Dtype
---  ------            --------------  -----
 0   Transaction ID    10000 non-null  str  
 1   Item              9031 non-null   str  
 2   Quantity          9521 non-null   str  
 3   Price Per Unit    9467 non-null   str  
 4   Total Spent       9498 non-null   str  
 5   Payment Method    6822 non-null   str  
 6   Location          6039 non-null   str  
 7   Transaction Date  9540 non-null   str  
dtypes: str(8)
memory usage: 625.1 KB


In [10]:
for col in ['Quantity', 'Price Per Unit', 'Total Spent']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

In [11]:
df.dtypes

Transaction ID          str
Item                    str
Quantity            float64
Price Per Unit      float64
Total Spent         float64
Payment Method          str
Location                str
Transaction Date        str
dtype: object

In [14]:
cols_chislo = df.select_dtypes(include=[np.number]).columns.tolist()
cols_text = [c for c in df.select_dtypes(include=['str']).columns if c != 'Transaction ID']
print(f"Числовые столбцы: {cols_chislo}")
print(f"Текстовые столбцы: {cols_text}")

Числовые столбцы: ['Quantity', 'Price Per Unit', 'Total Spent']
Текстовые столбцы: ['Item', 'Payment Method', 'Location', 'Transaction Date']


In [15]:
df['Transaction Date'] = pd.to_datetime(df['Transaction Date'], errors='coerce')
cols_chislo = df.select_dtypes(include=[np.number]).columns.tolist()
cols_date = df.select_dtypes(include=['datetime64[ns]']).columns.tolist()
cols_text = [c for c in df.select_dtypes(include=['object', 'str']).columns
if c != 'Transaction ID']
print(f"Числовые столбцы: {cols_chislo}")
print(f"Дата: {cols_date}")
print(f"Текстовые столбцы: {cols_text}")

Числовые столбцы: ['Quantity', 'Price Per Unit', 'Total Spent']
Дата: ['Transaction Date']
Текстовые столбцы: ['Item', 'Payment Method', 'Location']


In [20]:
df_before = df.copy()
for col in cols_chislo:
    mediana = df[col].median()
    df[col] = df[col].fillna(mediana)
    print(f"  {col}: медиана = {mediana:.2f}")

for col in cols_text:
    moda = df[col].mode()[0]
    df[col] = df[col].fillna(moda)
    print(f"  {col}: мода = {moda}")

for col in cols_date:
    mediana_date = df[col].median()
    df[col] = df[col].fillna(mediana_date)
    print(f"  {col}: медиана дат = {mediana_date}")
    
    print(f"Пропусков после заполнения: {df.isnull().sum().sum()}")
    

  Quantity: медиана = 3.00
  Price Per Unit: медиана = 3.00
  Total Spent: медиана = 8.00
  Item: мода = Juice
  Payment Method: мода = Digital Wallet
  Location: мода = Takeaway
  Transaction Date: медиана дат = 2023-07-02 00:00:00
Пропусков после заполнения: 0


In [21]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
df[cols_chislo] = scaler.fit_transform(df[cols_chislo])
df[cols_chislo].head()

,Quantity,Price Per Unit,Total Spent
0,0.25,0.25,0.125000
1,0.75,0.50,0.458333
2,0.75,0.00,0.291667
3,0.25,1.00,0.375000
4,0.25,0.25,0.125000


In [22]:
df = pd.get_dummies(df, columns=cols_text, drop_first=True)
df.head()

,Transaction ID,Quantity,Price Per Unit,Total Spent,Transaction Date,Item_Coffee,Item_Cookie,Item_Juice,Item_Salad,Item_Sandwich,Item_Smoothie,Item_Tea,Payment Method_Credit Card,Payment Method_Digital Wallet,Location_Takeaway
0,TXN_1961373,0.25,0.25,0.125000,2023-09-08,True,False,False,False,False,False,False,True,False,True
1,TXN_4977031,0.75,0.50,0.458333,2023-05-16,False,False,False,False,False,False,False,False,False,False
2,TXN_4271903,0.75,0.00,0.291667,2023-07-19,False,True,False,False,False,False,False,True,False,False
3,TXN_7034554,0.25,1.00,0.375000,2023-04-27,False,False,False,True,False,False,False,False,True,True
4,TXN_3160411,0.25,0.25,0.125000,2023-06-11,True,False,False,False,False,False,False,False,True,False


In [24]:
df.to_csv("dirty_cafe_sales_processed.csv", index=False)